# 🎥 Creator Shorts: Automated Podcast Highlights

This notebook automates the creation of short videos from podcasts using AI models. Optimized for Google Colab.

## 1. Setup & Upload
If you are running on Colab, upload the project files (`transcription.py`, `moment_selection.py`, etc.) and the video.

In [ ]:
# @title Upload Project Files (.py) and Video (.mp4)
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Upload all .py files and your video:")
    uploaded = files.upload()
else:
    print("Running locally. Ensure your Conda environment is active and files are in the same folder.")

In [ ]:
# @title Install Dependencies
if IN_COLAB:
    !apt-get install -y ffmpeg
    !pip install faster-whisper transformers accelerate moviepy opencv-python pillow
    print("Dependencies installed.")

## 2. Initialize Models

In [ ]:
import os
# Fix for OMP error on Windows/Colab
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from transcription import AudioTranscriber
from visual_analysis import VisualAnalyzer
from moment_selection import MomentSelector
from video_editor import VideoEditor
import torch

# @title Configuration
MODE = "Fast" # @param ["Fast", "High Quality"]
TRANSCRIPTION_MODEL = "large-v3" # @param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2" # @param ["mistralai/Mistral-7B-Instruct-v0.2", "meta-llama/Meta-Llama-3-8B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]
LLAVA_MODEL = "llava-hf/llava-1.5-7b-hf"

print(f"Initializing in {MODE} mode...")

transcriber = AudioTranscriber(model_size=TRANSCRIPTION_MODEL)
selector = MomentSelector(model_id=LLM_MODEL)
editor = VideoEditor()

analyzer = None
if MODE == "High Quality":
    analyzer = VisualAnalyzer(model_id=LLAVA_MODEL)

print("All models ready.")

## 3. Process Video

In [ ]:
# @title Run Pipeline
import json
video_path = "PODCAST.mp4" # @param {type:"string"}
output_dir = "saida_cortes"
os.makedirs(output_dir, exist_ok=True)

# Step 1: Transcribe
print("Step 1: Transcribing audio...")
transcription = transcriber.transcribe(video_path)

# Step 2: Visual Analysis (Optional)
visual_context = []
if MODE == "High Quality":
    print("Step 2: Visual analysis with LLaVA...")
    duration_s = transcription['segments'][-1]['end']
    for ts in range(0, int(duration_s), 10):
        frame = analyzer.extract_frame(video_path, ts * 1000)
        analysis = analyzer.analyze_frame(frame)
        visual_context.append({"timestamp": ts, "analysis": analysis})

# Step 3: Select Moments
print("Step 3: Finding viral moments...")
moments = selector.select_moments(transcription, visual_context if MODE == "High Quality" else None)
print(f"Found {len(moments)} moments.")

# Step 4: Clip and Detailed Metadata Export
print("Step 4: Generating clips and JSON...")
clip_paths = []
total_clips = len(moments)
for i, m in enumerate(moments):
    clip_name = f"clip_{i}"
    mp4_path = os.path.join(output_dir, f"{clip_name}.mp4")
    json_path = os.path.join(output_dir, f"{clip_name}.json")
    
    if editor.cut_video(video_path, mp4_path, m['start'], m['end']):
        clip_paths.append(mp4_path)
        
        # Detailed JSON
        clip_metadata = {
            "clip_id": i,
            "filenames": {"video": f"{clip_name}.mp4", "metadata": f"{clip_name}.json"},
            "timing": {"start": m['start'], "end": m['end'], "duration": m['end'] - m['start']},
            "analysis": {"reason": m['reason'], "climax_description": f"Gancho: {m['reason']}"},
            "batch_info": {"total_clips_generated": total_clips, "source_video": os.path.basename(video_path)}
        }
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(clip_metadata, f, indent=4, ensure_ascii=False)
        print(f"Generated: {mp4_path}")

print("Pipeline complete.")

## 4. Export & Compilation

In [ ]:
# @title Merge All Clips
merge_all = True # @param {type:"boolean"}

if merge_all and clip_paths:
    final_path = os.path.join(output_dir, "final_compilation.mp4")
    editor.merge_videos(clip_paths, final_path)
    print(f"Final compilation ready at: {final_path}")